# Final model interface example

This notebook runs the forecast-only LSTM, TCN, and XGBoost interfaces on the finalized feature table. The deep-learning examples use two epochs for a quick end-to-end check; no tuning is performed.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from IPython.display import display

from Final_models import train_lstm, train_tcn, train_xgboost

data = pd.read_csv(
    "../processed_data/finalized_feature_data_2024-03-01_2026-05-02.csv",
    parse_dates=["timestamp_utc"],
)
data.shape

(18988, 60)

In [2]:
lstm_config = {
    "lookback": 24,
    "hidden_dim": 64,
    "num_layers": 1,
    "dropout": 0.0,
    "learning_rate": 0.001,
    "focal_alpha": 0.5,
    "focal_gamma": 2.0,
    "seed": 1,
    "epochs": 2,
}

tcn_config = {
    "lookback": 48,
    "channels": (64, 64, 64, 64),
    "kernel_size": 3,
    "dropout": 0.3,
    "learning_rate": 0.001,
    "focal_alpha": 0.75,
    "focal_gamma": 2.0,
    "seed": 1,
    "epochs": 2,
}

xgboost_config = {
    "n_estimators": 300,
    "max_depth": 10,
    "min_child_weight": 1,
    "colsample_bytree": 0.3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "seed": 42,
}

In [3]:
def check_result(name, result, expected_shape):
    assert result["sequence_shape"] == expected_shape
    assert len(result["validation"]) == 3672
    assert len(result["test"]) == 3628

    for split in ["validation", "test"]:
        predictions = result[split]
        assert predictions["timestamp"].is_monotonic_increasing
        assert predictions[["y_true", "y_prob", "y_pred"]].notna().all().all()
        assert predictions["y_prob"].between(0, 1).all()
        assert set(predictions["y_pred"].unique()).issubset({0, 1})

    print(f"{name}: checks passed")
    display(result["metrics"])
    display(result["validation"].head())

## LSTM

The LSTM configuration is the current notebook default and can be replaced after the final LSTM tuning run.

In [4]:
lstm_result = train_lstm(data, lstm_config)
check_result("LSTM", lstm_result, (48, 42))

LSTM: checks passed


,PR_AUC,ROC_AUC,F1,precision,recall,threshold
split,,,,,,
validation,0.659272,0.965049,0.652956,0.619512,0.690217,0.734355
test,0.874926,0.988075,0.631148,0.974684,0.466667,0.734355


,timestamp,y_true,y_prob,y_pred
0,2025-07-01 00:00:00+00:00,0,0.084435,0
1,2025-07-01 01:00:00+00:00,0,0.097451,0
2,2025-07-01 02:00:00+00:00,0,0.112659,0
3,2025-07-01 03:00:00+00:00,0,0.129056,0
4,2025-07-01 04:00:00+00:00,0,0.148817,0


## TCN

In [5]:
tcn_result = train_tcn(data, tcn_config)
check_result("TCN", tcn_result, (72, 42))

TCN: checks passed


,PR_AUC,ROC_AUC,F1,precision,recall,threshold
split,,,,,,
validation,0.718611,0.975372,0.658537,0.597345,0.733696,0.819656
test,0.847978,0.988906,0.656604,0.870000,0.527273,0.819656


,timestamp,y_true,y_prob,y_pred
0,2025-07-01 00:00:00+00:00,0,0.064382,0
1,2025-07-01 01:00:00+00:00,0,0.084117,0
2,2025-07-01 02:00:00+00:00,0,0.096637,0
3,2025-07-01 03:00:00+00:00,0,0.138763,0
4,2025-07-01 04:00:00+00:00,0,0.150390,0


## XGBoost

In [6]:
xgboost_result = train_xgboost(data, xgboost_config)
check_result("XGBoost", xgboost_result, (90,))

XGBoost: checks passed


,PR_AUC,ROC_AUC,F1,precision,recall,threshold
split,,,,,,
validation,0.662738,0.973759,0.646091,0.519868,0.853261,0.223281
test,0.843440,0.986844,0.772277,0.847826,0.709091,0.223281


,timestamp,y_true,y_prob,y_pred
0,2025-07-01 00:00:00+00:00,0,0.000077,0
1,2025-07-01 01:00:00+00:00,0,0.000075,0
2,2025-07-01 02:00:00+00:00,0,0.000081,0
3,2025-07-01 03:00:00+00:00,0,0.000129,0
4,2025-07-01 04:00:00+00:00,0,0.000120,0


In [7]:
alignment = pd.DataFrame([
    {
        "model": "LSTM",
        "input_shape": lstm_result["sequence_shape"],
        "feature_count": len(lstm_result["feature_columns"]),
        "validation_rows": len(lstm_result["validation"]),
        "test_rows": len(lstm_result["test"]),
    },
    {
        "model": "TCN",
        "input_shape": tcn_result["sequence_shape"],
        "feature_count": len(tcn_result["feature_columns"]),
        "validation_rows": len(tcn_result["validation"]),
        "test_rows": len(tcn_result["test"]),
    },
    {
        "model": "XGBoost",
        "input_shape": xgboost_result["sequence_shape"],
        "feature_count": len(xgboost_result["feature_columns"]),
        "validation_rows": len(xgboost_result["validation"]),
        "test_rows": len(xgboost_result["test"]),
    },
]).set_index("model")
display(alignment)

,input_shape,feature_count,validation_rows,test_rows
model,,,,
LSTM,"(48, 42)",42,3672,3628
TCN,"(72, 42)",42,3672,3628
XGBoost,"(90,)",90,3672,3628


## Notebook alignment

With the same seed and two-epoch configuration, the LSTM and TCN validation probabilities and PR-AUC match their notebook implementations exactly. This interface uses exact validation thresholds for all models, so DL threshold/F1 may improve slightly over the notebooks' 0.01 threshold grid. XGBoost uses the same 90-feature recipe, but this finalized input starts at 2024-03-01 and therefore lacks 48 hours of pre-window SMARD history available to the raw-data notebook; its validation metric is checked with a small tolerance.

In [8]:
notebook_reference = pd.DataFrame({
    "LSTM": {"PR_AUC": 0.6592723707, "F1": 0.6479591837, "threshold": 0.73},
    "TCN": {"PR_AUC": 0.7186105516, "F1": 0.6552567237, "threshold": 0.82},
    "XGBoost": {"PR_AUC": 0.6684058889, "F1": 0.6418604651},
}).T

actual_validation = pd.DataFrame({
    "LSTM": lstm_result["metrics"].loc["validation"],
    "TCN": tcn_result["metrics"].loc["validation"],
    "XGBoost": xgboost_result["metrics"].loc["validation"],
}).T

for model_name in ["LSTM", "TCN"]:
    assert np.isclose(
        actual_validation.loc[model_name, "PR_AUC"],
        notebook_reference.loc[model_name, "PR_AUC"],
        atol=1e-4,
    )
    assert actual_validation.loc[model_name, "F1"] >= notebook_reference.loc[model_name, "F1"]

assert np.isclose(
    actual_validation.loc["XGBoost", "PR_AUC"],
    notebook_reference.loc["XGBoost", "PR_AUC"],
    atol=0.02,
)
assert np.isclose(
    actual_validation.loc["XGBoost", "F1"],
    notebook_reference.loc["XGBoost", "F1"],
    atol=0.02,
)

display(actual_validation[["PR_AUC", "F1", "threshold"]])
print("Notebook alignment checks passed")

,PR_AUC,F1,threshold
LSTM,0.659272,0.652956,0.734355
TCN,0.718611,0.658537,0.819656
XGBoost,0.662738,0.646091,0.223281


Notebook alignment checks passed
